# **Project**: Cross-Domain Aerial Detection and Segmentation: From Maritime to Urban, Desert and Mountain Environments

### Project Team members:

**Abdulmohsen Alharthi**

**Abdulelah Alkhamis**

**Ziyad Almutairi**

**Rayan Alsulami**

In [1]:
import sys, platform
print("Python:", sys.version.split()[0], "|", platform.platform())

Python: 3.13.16 | Linux-6.6.122+-x86_64-with-glibc2.39


In [2]:
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

PACKAGES = ["torch", "torchvision", "numpy", "pandas", "matplotlib", "scikit-learn", "opencv-python-headless",
            "transformers", "datasets", "ultralytics", "kagglehub", "wandb", "pyyaml"]

lines = []
for pkg in PACKAGES:
    try:
        lines.append(f"{pkg}=={version(pkg).split('+')[0]}")
    except PackageNotFoundError:
        pass
Path("requirements.txt").write_text("\n".join(lines) + "\n")
print(Path("requirements.txt").read_text())

torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
pandas==2.2.3
matplotlib==3.10.0
scikit-learn==1.6.1
opencv-python-headless==5.0.0.93
transformers==5.20.1
datasets==4.8.5
kagglehub==1.0.2
wandb==0.28.1
pyyaml==6.0.3



In [3]:
!pip -q install -r requirements.txt
print("installs done")

installs done


In [ ]:
# settings
import os, time, getpass

# ----  ------------------------------------------------------------------------------
USE_WANDB     = True
WANDB_ENTITY  = "rayan-alsulami-kaust"      #the team name
WANDB_PROJECT = "AI251PCapstoneProject"
DATASET       = #datasetName ""
TEST_TAG = # TAG NAME FOR TEST PROPUSE

SEED = 42
PROJECT_ROOT = Path("") #project_root
PROJECT_ROOT.mkdir(exist_ok=True)
os.chdir(PROJECT_ROOT)
print("config loaded | project root:", Path.cwd())

SyntaxError: invalid syntax (2404210362.py, line 13)

In [ ]:
# set seed
import random
import numpy as np, torch

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
print("seed set:", SEED)

In [ ]:
# Weights and Biases
import wandb

USE_WANDB = True

if USE_WANDB:
    if not os.environ.get("WANDB_API_KEY"):
        os.environ["WANDB_API_KEY"] = getpass.getpass("Enter W&B API Key: ")
    wandb.login()
    print(f"W&B Ready | Project: {WANDB_ENTITY}/{WANDB_PROJECT}")

In [ ]:
# naming and tags
TAGS = ["cls-scratch", "cls-frozen", "cls-full"]

def start_run(tag, model, lr, epochs, config=None):
    assert tag in TAGS, f"tag must be one of {TAGS}"
    name = f"{tag}_{model}_lr{lr:.0e}_e{epochs}_{time.strftime('%m%d-%H%M')}"
    if not USE_WANDB:
        return None
    return wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=name, group=tag,
                      tags=[tag, DATASET, model], config={"lr": lr, "epochs": epochs, "seed": SEED, **(config or {})})

In [ ]:
TAG   = ""            # fill: cls-scratch | cls-frozen | cls-full
MODEL = ""            # fill: e.g. "resnet50"

run = start_run(TAG, MODEL, lr=1e-3, epochs=3, extra_tags=[TEST_TAG])
for epoch in range(1, 4):
    if run:
        run.log({"epoch": epoch, "train/loss": 1 / epoch, "val/acc": 0.3 * epoch})
if run:
    run.finish()